# 1.0 — Fixed portfolio size + Fixed & Equal factor weights

In [ ]:
import sys, pathlib
SRC = pathlib.Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm

from paths import CLEAN_DATA, INTERIM_DATA
from metrics import total_return, ann_active_return
from chart import chart


# Load Data

In [ ]:
df_merge = pd.read_csv(CLEAN_DATA / '10y_merged.csv')
df_merge = df_merge.sort_values('date').set_index('date')


In [ ]:
df_factors = pd.read_csv(INTERIM_DATA / '10y_factors.csv')

factors = ['P/E', 'P/B', 'P/S', 'EV/EBITDA', 'FCF Yield', 'Earnings Yield']

df_factors[factors] = df_factors[factors].astype(float)
df_factors[factors] = stats.zscore(df_factors[factors])
zscore = df_factors.drop(columns=['12 Mo Yield', 'Company Id', 'SecId']).copy()
zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']] = - zscore[['P/E', 'P/B', 'P/S', 'EV/EBITDA']]


# Train / Test Split

In [ ]:
_n_ = 50

split = int(_n_ / 100 * len(df_merge))

df_train = df_merge.iloc[:split]
df_test  = df_merge.iloc[split:]

bmk_train = df_train['sprtrn']
bmk_test  = df_test['sprtrn']

rf_train = df_train['rf']
rf_test  = df_test['rf']


# Fixed portfolio size + Fixed & Equal factor weights

In [ ]:
port_size = 20

z_equal = zscore.copy()
df_m = df_test.copy()

# eqauly weighted z score
z_equal['score'] = z_equal[factors].mean(axis=1)     
z_equal = z_equal.sort_values('score', ascending=False).reset_index(drop=True)   

z_equal = z_equal[:port_size]
top_tic = z_equal['Ticker']

# portfolio monthly return
port_ret = df_m[top_tic].copy()
port_ret = port_ret.mean(axis=1)

display(top_tic)
chart(port_ret, bmk_test, rf_test)